# 03 · Baselines

Runs three existing sentiment tools on the 300-post test set and scores them with `src/evaluate.py`, the same scorer every later model uses. The authors' own tags are scored the same way, to see how often they match the hand labels.

- **Input:** `data/test_hand_labeled.csv` (from notebook 02)
- **Outputs:** one prediction file per model in `results/`: `preds_author_tag.csv`, `preds_vader.csv`, `preds_finbert.csv` and `preds_cryptobert.csv`

FinBERT and CryptoBERT download from Hugging Face the first time (about 1 GB in total). Both run on the CPU, which takes under a minute for 300 posts.

## Setup

In [1]:
import logging
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Hide two harmless notices printed when transformers is imported: one about notebook
# progress bars, and one from torch saying a feature we don't use isn't supported on macOS
warnings.filterwarnings("ignore", message="IProgress not found")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").addFilter(lambda record: False)
from huggingface_hub.utils import logging as hub_logging
from transformers import pipeline
from transformers.utils import logging as transformers_logging
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Let the notebook import our shared code from src/
ROOT = Path("..").resolve()
sys.path.append(str(ROOT))
from src.evaluate import evaluate, load_test, save_preds, score_posts, score_table
from src.labels import CRYPTOBERT_LABELS, FINBERT_LABELS, LABELS, vader_label

# Hide Hugging Face notices about login tokens and download progress; both models are public
hub_logging.set_verbosity_error()
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()

MODELS = ["author_tag", "vader", "finbert", "cryptobert"]

## How models are scored

Every model writes one file, `results/preds_<model>.csv`, with a predicted label for each test post. `src/evaluate.py` checks that the file covers all 300 posts, then compares it with the hand labels:

- **Confusion matrix:** a table that counts posts by true label (rows) and predicted label (columns). The diagonal holds the correct predictions. The other cells show which mistakes a model makes, such as calling bearish posts neutral.
- **Precision** for a label: of the posts the model *called* bullish, the share that really are bullish.
- **Recall** for a label: of the posts that *really are* bullish, the share the model found.
- **F1:** one score that combines precision and recall. It is only high when both are.
- **Macro-F1, the main metric:** the average of the three labels' F1 scores, so every label counts equally, however rare it is.
- **Invalid rate:** the share of outputs that aren't a label. This matters for the LLMs in later notebooks. An invalid output counts as wrong.

Every score is reported for all posts, for slang-heavy posts and for plain posts.

**Why macro-F1 instead of accuracy?** Over half of the test set is neutral, so a "model" that answers neutral every time looks decent on accuracy but not on macro-F1:

In [2]:
test = load_test()
always_neutral = score_posts(test["label"], ["neutral"] * len(test))
print(f"Always neutral: accuracy {always_neutral['accuracy']:.2f}, macro-F1 {always_neutral['macro_f1']:.2f}")

Always neutral: accuracy 0.53, macro-F1 0.23


## 1. Author tags

Before any model: how often do the authors' own tags match the hand labels? The tags are saved as if they were a model's predictions, so they go through the same scorer.

In [3]:
save_preds(test["post_id"], test["author_label"], "author_tag")

author = evaluate("author_tag")["all"]
print(f"Author tags match the hand labels on {author['accuracy']:.0%} of posts.")
author["confusion"].drop(columns="INVALID")  # tags are always a valid label

Author tags match the hand labels on 45% of posts.


predicted,bullish,bearish,neutral
true,,,
bullish,41,12,26
bearish,6,38,18
neutral,53,50,56


## 2. VADER

VADER (2014) is a rule-based tool built for social media. It has a dictionary of words with sentiment scores and rules for things like negation (*"not good"*) and emphasis (*"!!!"*). It combines them into a **compound score** from −1 (most negative) to +1 (most positive), and its standard cutoffs turn that into a label: ≥ 0.05 is bullish, ≤ −0.05 is bearish, and anything in between is neutral.

It knows everyday words like *"scam"*, but not most crypto slang, and not even *"bullish"*. When it gets slang right, it's often by luck: *"diamond hands"* scores positive only because *"diamond"* is a positive word in everyday English.

In [4]:
vader = SentimentIntensityAnalyzer()

for text in ["diamond hands", "hodl", "got rekt again", "this is going to moon", "very bullish", "this coin is a scam"]:
    print(f"{vader.polarity_scores(text)['compound']:+.2f}  {text}")

+0.34  diamond hands
+0.00  hodl
+0.00  got rekt again
+0.00  this is going to moon
+0.00  very bullish
-0.57  this coin is a scam


In [5]:
compound = [vader.polarity_scores(text)["compound"] for text in test["text"]]
save_preds(test["post_id"], [vader_label(score) for score in compound], "vader")

## 3. FinBERT

FinBERT (`ProsusAI/finbert`) is a BERT model fine-tuned on sentences from financial news. It predicts positive, negative or neutral, which map to bullish, bearish and neutral.

A Hugging Face **pipeline** bundles the two steps every transformer model needs: the tokenizer, which splits text into the word pieces the model knows, and the model itself, which scores each label.

In [6]:
finbert = pipeline("text-classification", model="ProsusAI/finbert")
outputs = finbert(test["text"].tolist(), batch_size=16, truncation=True)
save_preds(test["post_id"], [FINBERT_LABELS[out["label"]] for out in outputs], "finbert")

## 4. CryptoBERT

CryptoBERT (`ElKulako/cryptobert`) starts from BERTweet, a model trained on tweets, and was fine-tuned on about 2M StockTwits crypto posts to predict Bullish, Bearish or Neutral.

**Caveat:** those posts come from the same StockTwits dataset as ours, and its training targets were the same author tags. Some test posts may even have been in its training data, so its score is likely optimistic.

Its model card says it was trained on posts of up to 128 tokens, so longer posts are cut at 128:

In [7]:
cryptobert = pipeline("text-classification", model="ElKulako/cryptobert")

lengths = [len(cryptobert.tokenizer(text)["input_ids"]) for text in test["text"]]
print(f"{sum(n > 128 for n in lengths)} of {len(test)} posts are longer than 128 tokens")

outputs = cryptobert(test["text"].tolist(), batch_size=16, truncation=True, max_length=128)
save_preds(test["post_id"], [CRYPTOBERT_LABELS[out["label"]] for out in outputs], "cryptobert")

1 of 300 posts are longer than 128 tokens


## 5. Results

Macro-F1 for each model, on all posts and on each group:

In [8]:
table = score_table(MODELS)
table.pivot(index="model", columns="group", values="macro_f1").loc[MODELS, ["all", "slang-heavy", "plain"]].round(3)

group,all,slang-heavy,plain
model,,,
author_tag,0.453,0.502,0.403
vader,0.396,0.402,0.382
finbert,0.366,0.298,0.421
cryptobert,0.466,0.475,0.441


CryptoBERT's score is likely optimistic (see section 4). None of these models produce invalid outputs, since each one always returns one of its labels.

F1 for each label, on all posts:

In [9]:
pd.DataFrame({model: evaluate(model)["all"]["per_class"]["f1"] for model in MODELS}).T.loc[MODELS, LABELS].round(3)

label,bullish,bearish,neutral
author_tag,0.458,0.469,0.432
vader,0.368,0.395,0.426
finbert,0.136,0.294,0.668
cryptobert,0.559,0.480,0.358


Confusion matrices on all posts (rows are the hand labels, columns the predictions):

In [10]:
for model in ["vader", "finbert", "cryptobert"]:
    print(model)
    display(evaluate(model)["all"]["confusion"].drop(columns="INVALID"))

vader


predicted,bullish,bearish,neutral
true,,,
bullish,34,14,31
bearish,15,30,17
neutral,57,46,56


finbert


predicted,bullish,bearish,neutral
true,,,
bullish,6,4,69
bearish,2,15,45
neutral,1,21,137


cryptobert


predicted,bullish,bearish,neutral
true,,,
bullish,59,13,7
bearish,8,41,13
neutral,65,55,39


## Summary

- **The author tags match the hand labels on only 45% of posts** (macro-F1 0.45). The main reason is that authors tagged 103 of the 159 posts that read as neutral by hand as bullish or bearish. This is the label noise the fine-tuned model will learn from.
- **CryptoBERT scores best (0.47), but it behaves like the author tags.** It was trained on those tags, and it rarely predicts neutral: only 39 of the 159 neutral posts. Its score is also likely optimistic, since it was trained on the same StockTwits data.
- **VADER (0.40) does about the same on slang-heavy and plain posts.** It misses most slang, but plain crypto posts are hard for it too, since it doesn't even know *"bullish"*.
- **FinBERT (0.37) drops on slang-heavy posts (0.30 vs. 0.42 on plain)** and calls most posts neutral. It finds only 6 of the 79 bullish posts, likely because financial news sounds nothing like crypto posts.
- All three beat the always-neutral baseline (0.23), but none passes 0.5. That leaves plenty of room for the LLM in notebooks 04 and 05.